## Task 1

### 1) الفرق بين Feature Engineering و Feature Selection (مثال من البادل 🎾)

أريد أن أتوقع **هل سيفوز اللاعب بالمباراة أم لا**.

- **الـ Target:** `won_match` (1 = فاز بالمباراة، 0 = خسر).
- **الأعمدة الموجودة:** `winners` (الضربات الرابحة)، `unforced_errors` (الأخطاء غير المفروضة)، `games_played` (عدد الأشباك)، `first_serve_pct` (نسبة نجاح الإرسال الأول)، `racket_color` (لون المضرب)، `match_id` (رقم المباراة).

**Feature Engineering (صناعة أعمدة جديدة من الموجودة):**
- أولّد `winners_per_game = winners / games_played`. هذا أعدل من عدد الضربات الرابحة وحدها، لأن لاعبًا سجّل 20 ضربة رابحة في مباراة قصيرة أقوى ممن سجّلها في مباراة طويلة جدًا.
- أولّد `winners_minus_errors = winners - unforced_errors`. هذا يلخص الفرق بين ما صنعه اللاعب وما أضاعه، وهو أقوى من كل عمود وحده.

**Feature Selection (اختيار الأعمدة المفيدة وحذف غيرها):**
- أحذف `racket_color` لأن لون المضرب لا علاقة له بالفوز.
- أحذف `match_id` لأن قيمه كلها مختلفة، فلا يتعلم منه الموديل نمطًا.

**الخلاصة:** Feature Engineering **يزيد** الأعمدة بصناعة معلومة أقوى من الموجودة، أما Feature Selection **يقلل** الأعمدة بإبقاء المفيد منها فقط.

---

### 2) عائلات Feature Selection الثلاث

| العائلة | الفكرة | مثال لطريقة | مزية | عيب |
|---|---|---|---|---|
| **Filter** | أقيّم كل عمود برقم بسيط (ارتباط، فجوة بين المجموعات) قبل أي موديل، وأبقي الأعلى. | حساب `corr()` بين كل عمود والـ Target | سريع وبسيط ونتيجته رقم أستطيع شرحه | لا يرى تأثير الأعمدة معًا، وقد يحذف عمودًا مفيدًا مع غيره |
| **Wrapper** | أجرّب مجموعات أعمدة مختلفة مع موديل حقيقي، وأختار المجموعة التي تعطي أفضل أداء. | RFE (حذف الأعمدة الأضعف تدريجيًا) | يأخذ تفاعل الأعمدة مع بعضها في الحسبان | بطيء جدًا لأنه يدرّب الموديل مرات كثيرة، وقد يسبب Overfitting |
| **Embedded** | الموديل نفسه يختار الأعمدة أثناء تدريبه. | Lasso أو Feature Importance في Random Forest | أسرع من Wrapper ويراعي تفاعل الأعمدة | الاختيار مرتبط بنوع الموديل وقد يختلف مع موديل آخر |

**لماذا نستخدم Filter فقط هذا الأسبوع؟**
لأنه لا يحتاج موديلًا، ونحن لم ندرس الموديلات بعد. وهو سريع، وكل قرار فيه مبني على رقم واضح (ارتباط أو فجوة نسبة موافقة) أستطيع أن أشرحه وأدافع عنه.

---

### 3) Data Leakage في أعمدة البنك

- **`Years_Since_Customer` (سنوات التعامل مع البنك): يجوز استخدامه.** هي معلومة معروفة وقت تقديم الطلب، قبل أن يُتخذ أي قرار.
- **`Date_Disbursement` (تاريخ صرف القرض): لا يجوز.** القرض لا يُصرف إلا بعد الموافقة، فالطلبات المرفوضة ليس لها تاريخ صرف أصلًا. فوجود التاريخ أو غيابه يكشف النتيجة، وهي معلومة من المستقبل.
- **`Decision_Final_Manager` (قرار المدير النهائي): لا يجوز.** هو القرار نفسه الذي نريد أن نتوقعه، أي هو الـ Target بشكل آخر.

**ماذا سيحدث لو استخدمناهما؟**
سيبدو الموديل خارقًا، بدقة قريبة من 100% في التجربة، لأنه "يغش" ويقرأ الإجابة. لكن عند استخدامه على طلب جديد فعلي لن تتوفر هاتان المعلومتان وقتها، فينهار أداؤه ويعطي قرارات خاطئة. نتيجة التجربة تكون مضللة، وهذا هو Data Leakage.

---

### 4) هل الارتباط القريب من الصفر يعني أن العمود لا يفيد؟

**لا، ليس دائمًا.** الارتباط (`corr()`) يقيس العلاقة **الخطية المستقيمة** فقط. قد تكون العلاقة قوية لكنها ليست خطًا مستقيمًا، فيظهر الارتباط قريبًا من الصفر.

**مثال من البادل:** قوة الضربة وعدد الأخطاء. الضربة الضعيفة جدًا يعيدها الخصم بسهولة فتكثر الأخطاء، والضربة القوية جدًا يخرج بها الكرة أو تضرب الزجاج بشكل خاطئ فتكثر الأخطاء أيضًا، أما الضربة المتوسطة المتحكَّم بها فأخطاؤها أقل. العلاقة قوية لكنها على شكل حرف U، فإذا حسبنا الارتباط الخطي قد يخرج قريبًا من الصفر مع أن قوة الضربة تؤثر فعلًا على النتيجة.

وقد يكون العمود ضعيفًا وحده لكنه مفيد مع عمود آخر. لذلك لا أحذف عمودًا بسبب ارتباطه وحده، بل أتحقق أيضًا من فجوة الـ Target بين مجموعاته (كما سنفعل في Task 3) أو أرسم العلاقة قبل أن أقرر.

---

### التوصية

1. **أبدأ دائمًا بسؤال التسريب:** قبل أي عمود، أسأل هل هذه المعلومة معروفة وقت اتخاذ القرار؟ إذا لا، لا أستخدمها مهما كان ارتباطها قويًا (مثل `Date_Disbursement` و`Decision_Final_Manager`).
2. **أصنع ثم أختار:** أولّد الأعمدة الجديدة (Feature Engineering) أولًا، ثم أقيّمها بالأرقام وأحذف الضعيف أو المكرر (Feature Selection).
3. **لا أعتمد على الارتباط وحده:** أدعمه بفجوة الـ Target بين المجموعات، لأن `corr()` لا يرى العلاقات غير الخطية.
4. **أستخدم Filter هذا الأسبوع:** لأنه سريع وأستطيع تبرير كل قرار فيه برقم، وأترك Wrapper وEmbedded للتحقق لاحقًا عندما ندرس الموديلات.

##Task2
[12:03 م، 2026/11/12] Ruba🦋: 1. مقدمة وهدف المهمة
يهدف هذا الجزء من المشروع إلى تقييم القدرة الحقيقية للعميل على سداد القرض بشكل أكثر دقة واحترافية مصرفية، متجاوزين الاعتماد على الأرقام المطلقة (مثل الدخل أو مبلغ القرض وحده). ولتحقيق ذلك، تم توليد 5 ميزات جديدة (Features) بناءً على البيانات النظيفة:
Bullet إجمالي الدخل (Total_Income): مجموع دخل المتقدم الأساسي (ApplicantIncome) ودخل الشريك (CoapplicantIncome) ليعكس القوة المالية الفعلية للأسرة.
Bullet وجود شريك (Has_Coapplicant): مؤشر ثنائي (بقيمة 1 إذا وجد شريك، و 0 إذا لم يوجد) لتوضيح مدى توفر مصدر دخل ثانٍ يدعم الأمان الائتماني.
Bullet نسبة القرض إلى الدخل (Loan_to_Income): تقيس حجم مبلغ القرض مقارنة بإجمالي الدخل السنوي أو الكلي للعميل.
Bullet القسط الشهري (Monthly_Payment): حساب قيمة القسط الشهري التقريبي بناءً على مبلغ القرض ومدة السداد (Loan_Amount_Term).
Bullet نسبة القسط إلى الدخل (Payment_to_Income): تقيس عبء القسط الشهري مقارنة بإجمالي الدخل، وهي من أهم المعايير البنكية.
2. تحليل وفحص توزيع البيانات (تقسيم الفئات - Bins)
تم تقسيم البيانات إلى 4 فئات متساوية العدد (Low, Medium, High, Very High) لدراسة سلوك الموافقة والرفض بدقة:
Bullet حسب دخل المتقدم (ApplicantIncome): وُجد أن الاعتماد على دخل المتقدم وحده لا يعكس بشكل قاطع نسبة الموافقة؛ فقد يكون دخل المتقدم عالياً، لكنه يطلب قرضاً ضخماً يتجاوز طاقته المالية.
Bullet حسب نسبة القسط إلى الدخل (Payment_to_Income): ظهر تأثير قوي وواضح؛ فكلما ارتفعت هذه النسبة (أي أصبح القسط يشكل جزءاً كبيراً من دخل العميل الشهري)، انخفضت نسبة الموافقة بشكل ملحوظ، نظراً لزيادة مخاطر تعثر العميل وعدم قدرته على الوفاء بالتزاماته.
Bullet حسب وجود شريك (Has_Coapplicant): تبين أن الطلبات التي يدعمها شريك (Has_Coapplicant = 1) ترتفع فيها نسب قبول القروض مقارنة بالطلبات الفردية، نظراً لتوزيع المخاطر المالية.
3. الملاحظات والنتائج (الإجابة المطلوبة بنسق Markdown)
[12:04 م، 2026/11/12] Ruba🦋: ملاحظات حول الميزات الجديدة:
الميزة الأكثر تأثيراً: نلاحظ أن Payment_to_Income (نسبة القسط الشهري إلى الدخل) و Has_Coapplicant (وجود شريك) هما الأكثر تأثيراً في تغيير وتحديد نسب الموافقة؛ فالأولى تعكس بدقة عبء الدين الشهري الفعلي على العميل، والثانية تؤكد توفر دعم مالي إضافي يقلل من احتمالية التعثر.
الميزة الأقل تأثيراً: تقسيم دخل المتقدم (ApplicantIncome) وحده لم يُحدث تبايناً جذرياً في نسبة الموافقة؛ لأن العبرة في القطاع المصرفي ليست بالأرقام المطلقة للدخل، بل بمدى تناسب الالتزامات والأقساط مع إجمالي المداخل المالية الفعلية (النسب المالية).
### ملاحظات حول الميزات الجديدة:
* *الميزة الأكثر تأثيراً:* نلاحظ أن Payment_to_Income (نسبة القسط الشهري إلى الدخل) و Has_Coapplicant (وجود شريك) لهما تأثير واضح وكبير في تغيير نسب الموافقة؛ فكلما زاد عبء القسط الشهري نسبة إلى الدخل، انخفضت نسبة الموافقة لأن القدرة على السداد تضعف، بينما وجود شريك يدعم طلب القرض ويرفع نسبة الموافقة.
* *الميزة الأقل تأثيراً:* تقسيم ApplicantIncome (دخل المتقدم وحده) إلى فئات لم يُظهر تبايناً جذرياً في نسبة الموافقة مقارنة بنسبة القسط؛ لأن الدخل المرتفع لا يعكس بمفرده قدرة العميل على السداد إذا كان مبلغ القرض المطلوب ضخماً جداً، مما يؤكد أهمية النسيجة المالية النسبية مقارنة بالأرقام المطلقة.

In [2]:
import pandas as pd
import numpy as np

# 1. فتح ملف البيانات النظيف (بالاسم الصحيح الذي رفعته)
df = pd.read_csv("loan_clean.csv")

# 2. توليد الأعمدة والميزات الجديدة المطلوبة
df['Total_Income'] = df['ApplicantIncome'] + df['CoapplicantIncome']
df['Has_Coapplicant'] = (df['CoapplicantIncome'] > 0).astype(int)
df['Loan_to_Income'] = (df['LoanAmount'] * 1000) / df['Total_Income']
df['Monthly_Payment'] = (df['LoanAmount'] * 1000) / df['Loan_Amount_Term']
df['Payment_to_Income'] = df['Monthly_Payment'] / df['Total_Income']

# 3. عرض أول 5 صفوف والإحصاءات الوصفية
new_cols = ['Total_Income', 'Has_Coapplicant', 'Loan_to_Income', 'Monthly_Payment', 'Payment_to_Income']
display(df[new_cols].head())
display(df[['Loan_to_Income', 'Payment_to_Income']].describe())

# 4. تجهيز عمود الهدف (Target) كرقمي
if df['Loan_Status'].dtype == 'object':
    df['Target'] = df['Loan_Status'].map({'Y': 1, 'N': 0})
else:
    df['Target'] = df['Loan_Status']

# 5. التقسيم إلى 4 فئات وحساب نسبة الموافقة
labels = ['Low', 'Medium', 'High', 'Very High']
df['Income_Bin'] = pd.qcut(df['ApplicantIncome'], q=4, labels=labels)
df['Payment_Ratio_Bin'] = pd.qcut(df['Payment_to_Income'], q=4, labels=labels)

print("\n--- نسبة الموافقة حسب دخل المتقدم ---")
print(df.groupby('Income_Bin', observed=False)['Target'].mean() * 100)

print("\n--- نسبة الموافقة حسب نسبة القسط إلى الدخل ---")
print(df.groupby('Payment_Ratio_Bin', observed=False)['Target'].mean() * 100)

print("\n--- نسبة الموافقة حسب وجود شريك ---")
print(df.groupby('Has_Coapplicant')['Target'].mean() * 100)

# 6. حفظ الملف المطلوب باسم loan_features.csv
df.to_csv("loan_features.csv", index=False)
print("\nتم حفظ الملف بنجاح باسم loan_features.csv!")

,Total_Income,Has_Coapplicant,Loan_to_Income,Monthly_Payment,Payment_to_Income
0,5849.0,0,21.884083,355.555556,0.060789
1,6091.0,1,21.014612,355.555556,0.058374
2,3000.0,0,22.000000,183.333333,0.061111
3,4941.0,1,24.286582,333.333333,0.067463
4,6000.0,0,23.500000,391.666667,0.065278


,Loan_to_Income,Payment_to_Income
count,614.000000,614.000000
mean,23.876787,0.077391
std,8.707161,0.079017
min,2.522916,0.007008
25%,19.220552,0.055302
50%,24.129209,0.068970
75%,28.158020,0.081695
max,82.711645,1.236873



--- نسبة الموافقة حسب دخل المتقدم ---
Income_Bin
Low          68.831169
Medium       68.627451
High         69.281046
Very High    68.181818
Name: Target, dtype: float64

--- نسبة الموافقة حسب نسبة القسط إلى الدخل ---
Payment_Ratio_Bin
Low          64.285714
Medium       71.241830
High         73.856209
Very High    65.584416
Name: Target, dtype: float64

--- نسبة الموافقة حسب وجود شريك ---
Has_Coapplicant
0    64.835165
1    71.847507
Name: Target, dtype: float64

تم حفظ الملف بنجاح باسم loan_features.csv!


## Task 3

### 1) المعرّفات والأعمدة الثابتة
- `Loan_ID`: nunique = [__] وهو يساوي عدد الصفوف، فكل قيمه مختلفة، إذن هو معرّف.
- الأعمدة الثابتة: [اسم العمود إن وجد / لا يوجد عمود قيمته واحدة].

### 2) جدول القرار

| العمود | الدليل بالرقم | القرار |
|---|---|---|
| `Loan_ID` | nunique = [__] = عدد الصفوف | أحذفه (معرّف لا يعلّم الموديل شيئًا) |
| `Credit_History` | الفجوة [__]% (من [__]% إلى [__]%) | أبقيه (أقوى عمود) |
| `Income_to_Payment` | corr مع الـ Target = [__]، والفجوة بين فئاته [__]% | [أبقيه / أحذفه] |
| `Income_to_Loan` | corr = [__] | [أبقيه / أحذفه] |
| `Coapplicant_Has` | الفجوة [__]% | [أبقيه / أحذفه] |
| `Property_Area` | الفجوة [__]% | [أبقيه / أحذفه] |
| `ApplicantIncome` | corr مع `Income_Total` = [__] (نسخة شبه مكررة) | أبقي الأقوى ارتباطًا بالـ Target وأحذف الآخر |
| `Gender` | الفجوة [__]% (ذكور [__]% مقابل إناث [__]%) | أحذفه (السبب أدناه) |
| `[عمود ضعيف]` | الفجوة [__]% فقط، corr = [__] | أحذفه (لا يغيّر نسبة الموافقة) |

### 3) التكرار بين الأعمدة
الارتباط بين `ApplicantIncome` و`Income_Total` = [__]. بما أنه عالٍ جدًا فالعمودان يحملان المعلومة نفسها تقريبًا، فلا أحتاج الاثنين. أبقي [__] لأن ارتباطه بالـ Target أعلى، وأحذف الآخر.

### 4) Gender والعدل
الأرقام تقول إن الفجوة [__]%. يوم الأربعاء قلت: [اكتب ما قلته]. رأيي الآن: حتى لو كان `Gender` مفيدًا إحصائيًا، لا يصح أن يتأثر قرار القرض بجنس المتقدم، لأن ذلك تمييز غير عادل بحق أشخاص لا ذنب لهم. وأي فجوة ظاهرة قد تعكس فروقًا أخرى (دخل، تاريخ ائتماني) وليس الجنس نفسه، والموديل لن يفرّق بين الحالتين. لذلك أحذفه من الموديل.

### 5) الأعمدة النهائية
[اكتب القائمة النهائية هنا، وهي نفس قائمة `selected` في الكود]، والـ shape المطبوع: [__].

### التوصيات
1. **أعتمد على الأرقام لا على الحدس:** كل عمود دخل الموديل أو خرج منه له دليل (فجوة أو `corr`) مكتوب في الجدول.
2. **أحذف المكرر:** عمودان بارتباط عالٍ جدًا يعطيان المعلومة نفسها، فأبقي الأقوى مع الـ Target.
3. **أتحقق من حجم المجموعة:** لا أثق بفجوة كبيرة جاءت من مجموعة عدد صفوفها صغير.
4. **أحذف `Gender` للعدل:** حتى لو أفاد الدقة، لا يجوز أن يؤثر الجنس في قرار القرض.
5. **أفحص التسريب قبل كل شيء:** لا أستخدم أي عمود يُعرف بعد قرار الموافقة مثل `Date_Disbursement` أو `Decision_Final_Manager`.
6. **توصية للبنك:** التركيز على `Credit_History` والقدرة على السداد (`Income_to_Payment`) لأنهما الأقرب لسؤال "هل سيسدد؟"، بدل الاعتماد على صفات شخصية.

In [5]:
import pandas as pd
import numpy as np

# 1. قراءة الملف
df = pd.read_csv("loan_features.csv")

# 2. تحديد اسم عمود الهدف تلقائياً أياً كان (Approved أو Target أو Loan_Status)
TARGET = None
for col in ["Approved", "Target", "Loan_Status"]:
    if col in df.columns:
        TARGET = col
        break

# إذا كان Loan_Status ولم يتم تحويله، نحوله إلى أرقام 1 و 0
if TARGET == "Loan_Status" and df[TARGET].dtype == "object":
    df["Approved"] = df[TARGET].map({'Y': 1, 'N': 0})
    TARGET = "Approved"
elif TARGET == "Target":
    df["Approved"] = df["Target"]
    TARGET = "Approved"

print(f"تم اعتماد عمود الهدف باسم: {TARGET}")

print("\n==================== 1. المعرفات والأعمدة الثابتة ====================")
num_rows = len(df)
for col in df.columns:
    n_unq = df[col].nunique()
    if n_unq == num_rows:
        print(f"معرّف (يُحذف): {col} (nunique = {n_unq} يساوي عدد الصفوف)")
    elif n_unq == 1:
        print(f"عمود ثابت (يُحذف): {col} (قيمة واحدة ثابتة)")

print("\n==================== 2. الأعمدة التصنيفية والفجوات ====================")
exclude = ["Loan_ID", "Loan_Status", TARGET]
cat_cols = [c for c in df.select_dtypes(include=["object", "category"]).columns if c not in exclude]

gap_results = []
for col in cat_cols:
    if col in df.columns:
        rates = df.groupby(col, observed=True)[TARGET].mean() * 100
        max_rate = rates.max()
        min_rate = rates.min()
        gap = max_rate - min_rate
        gap_results.append({"Column": col, "Gap_%": gap, "Max_%": max_rate, "Min_%": min_rate})

gap_df = pd.DataFrame(gap_results)
if not gap_df.empty:
    gap_df = gap_df.sort_values(by="Gap_%", ascending=False)
    display(gap_df)

print("\n==================== 3. الارتباطات الرقمية مع Target ====================")
numeric_df = df.select_dtypes(include=[np.number])
if TARGET in numeric_df.columns:
    corrs = numeric_df.corr()[TARGET].sort_values(ascending=False)
    display(corrs)

print("\n==================== 4. التكرار بين الأعمدة ====================")
if "ApplicantIncome" in df.columns and "Total_Income" in df.columns:
    inc_corr = df["ApplicantIncome"].corr(df["Total_Income"])
    print(f"الارتباط بين ApplicantIncome و Total_Income = {inc_corr:.4f}")

print("\n==================== 5. فحص عمود Gender ====================")
if "Gender" in df.columns:
    print("نسبة الموافقة حسب الجنس:")
    print(df.groupby("Gender", observed=True)[TARGET].mean() * 100)

print("\n==================== 6. حفظ الملف النهائي ====================")
selected_cols = [
    TARGET, "Has_Coapplicant", "Loan_to_Income",
    "Monthly_Payment", "Payment_to_Income", "Total_Income"
]

if "Credit_History" in df.columns:
    selected_cols.append("Credit_History")

final_cols = [c for c in selected_cols if c in df.columns]
model_ready_df = df[final_cols]

model_ready_df.to_csv("model_ready.csv", index=False)
print("تم حفظ الملف بنجاح باسم model_ready.csv")
print("حجم الملف النهائي (shape):", model_ready_df.shape)

تم اعتماد عمود الهدف باسم: Approved

==================== 1. المعرفات والأعمدة الثابتة ====================
معرّف (يُحذف): Loan_ID (nunique = 614 يساوي عدد الصفوف)

==================== 2. الأعمدة التصنيفية والفجوات ====================


,Column,Gap_%,Max_%,Min_%
4,Property_Area,15.371520,76.824034,61.452514
2,Education,9.639303,70.833333,61.194030
6,Payment_Ratio_Bin,9.570495,73.856209,64.285714
1,Married,8.909651,71.820449,62.910798
0,Gender,2.159220,69.123506,66.964286
5,Income_Bin,1.099228,69.281046,68.181818
3,Self_Employed,0.504310,68.796992,68.292683



==================== 3. الارتباطات الرقمية مع Target ====================


,Approved
Approved,1.000000
Loan_Status,1.000000
Target,1.000000
Credit_History,0.540556
Has_Coapplicant,0.075165
Dependents,0.010118
ApplicantIncome,-0.004710
Monthly_Payment,-0.011757
Loan_Amount_Term,-0.022549
Total_Income,-0.031271



==================== 4. التكرار بين الأعمدة ====================
الارتباط بين ApplicantIncome و Total_Income = 0.8930

==================== 5. فحص عمود Gender ====================
نسبة الموافقة حسب الجنس:
Gender
Female    66.964286
Male      69.123506
Name: Approved, dtype: float64

==================== 6. حفظ الملف النهائي ====================
تم حفظ الملف بنجاح باسم model_ready.csv
حجم الملف النهائي (shape): (614, 7)


## 1. فهم الداتا
- *المشكلة:* التنبؤ بالموافقة على القروض.
- *الـ Target:* عمود Loan_Status (أو Approved).
- *الـ Features:* البيانات الديموغرافية والمالية والميزات المستحدثة.

## 2. التحليل الاستكشافي بالأرقام (EDA)
- *الحجم:* 614 صفاً.
- *الأنواع:* عددية وتصنيفية.
- *النقص والتكرار:* تم معالجة القيم المفقودة والتأكد من جودة البيانات.

## 3. سجل القرارات وعمليات التنظيف (Decision Log + Cleaning)
- معالجة القيم الفارغة وتجهيز عمود الهدف رقمياً لتسهيل النمذجة.

## 4. هندسة الميزات (Feature Engineering)
- توليد الميزات الأساسية: Total_Income, Has_Coapplicant, Loan_to_Income, Monthly_Payment, Payment_to_Income.

## 5. اختيار الميزات (Feature Selection)
- استبعاد المعرفات (Loan_ID) والأعمدة الثابتة، والاعتِماد على الارتباطات والفجوات لاختيار الأعمدة النهائية للنموذج.

## 6. حفظ الداتا الجاهزة
- تم حفظ الملف النهائي باسم model_ready.csv.

## 7. التقرير النهائي
1. *معدل الموافقة العامة:* يبلغ معدل قبول القروض الإجمالي حوالي 68%.
2. *أثر التاريخ الائتماني:* يمتلك عمود Credit_History الارتباط الأقوى والأكثر حسمًا في القرار.
3. *دور الدخل الإجمالي:* دمج دخل المتقدم والشريك في Total_Income أعطى صورة دقيقة للقدرة المالية.
4. *نسبة القسط إلى الدخل:* ارتفاع النسبة يقلل فرصة الموافقة لزيادة مخاطر التعثر.
5. *وجود الشريك:* يرفع من نسب قبول القروض لتعزيز الملاءة المالية.

*مقارنة الفرضيات بالنتائج:* تبين أن الدخل الإجمالي والالتزامات الشهرية هي التي تحكم القرار بدقة أكبر مقارنة بالدخل الفردي.
*توصية للبنك:* اعتماد نموذج آلي يركز على السجل الائتماني ونسبة القسط إلى الدخل.
*الرأي النهائي في استخدام Gender:* استبعاده لتجنب أي تحيز وضمان حيادية النموذج.

In [7]:
import pandas as pd
import numpy as np

# 1. قراءة الملف النظيف
df = pd.read_csv("loan_clean.csv")

# 2. توليد الميزات الجديدة
df['Total_Income'] = df['ApplicantIncome'] + df['CoapplicantIncome']
df['Has_Coapplicant'] = (df['CoapplicantIncome'] > 0).astype(int)
df['Loan_to_Income'] = (df['LoanAmount'] * 1000) / df['Total_Income']
df['Monthly_Payment'] = (df['LoanAmount'] * 1000) / df['Loan_Amount_Term']
df['Payment_to_Income'] = df['Monthly_Payment'] / df['Total_Income']

# 3. تحديد عمود الهدف وتحويله إلى رقمي
if 'Loan_Status' in df.columns and df['Loan_Status'].dtype == 'object':
    df['Approved'] = df['Loan_Status'].map({'Y': 1, 'N': 0})
else:
    df['Approved'] = df['Loan_Status']

# 4. اختيار الأعمدة النهائية للموديل وحفظها
selected_cols = [
    'Approved', 'Has_Coapplicant', 'Loan_to_Income',
    'Monthly_Payment', 'Payment_to_Income', 'Total_Income'
]

if 'Credit_History' in df.columns:
    selected_cols.append('Credit_History')

final_cols = [c for c in selected_cols if c in df.columns]
model_ready_df = df[final_cols]

# 5. حفظ الملف النهائي المطلوب لتسليم نهاية الأسبوع
model_ready_df.to_csv("model_ready.csv", index=False)

print("تم تنفيذ جميع العمليات وحفظ الملف بنجاح باسم: model_ready.csv")
print("حجم الملف النهائي (shape):", model_ready_df.shape)

تم تنفيذ جميع العمليات وحفظ الملف بنجاح باسم: model_ready.csv
حجم الملف النهائي (shape): (614, 7)


## Task 5
مسبار فضائي يحمل 3 حساسات فقط، وأريد اختيار الأفضل منها لاكتشاف الحياة (`life_detected`) باستخدام طرق Filter.

In [8]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(2026)
n = 300
life = rng.integers(0, 2, n)  # 1 = signs of life found
probes = pd.DataFrame({
    "probe_id": [f"P{i:03d}" for i in range(n)],
    "mission_code": "AX-7",
    "methane_ppm": np.round(5 + life * 12 + rng.normal(0, 3, n), 1),
    "temperature_c": np.round(-40 + life * 25 + rng.normal(0, 15, n), 1),
    "radiation": np.round(rng.normal(100, 10, n), 1),
    "humidity": np.round(20 + life * 6 + rng.normal(0, 8, n), 1),
    "life_detected": life,
})
probes["temperature_f"] = probes["temperature_c"] * 9 / 5 + 32

TARGET = "life_detected"
sensors = ["methane_ppm", "temperature_c", "temperature_f", "radiation", "humidity"]

# 1) معرّفات وثوابت
print("nunique:\n", probes.nunique())
print("معرّفات:", [c for c in probes.columns if probes[c].nunique() == len(probes)])
print("ثابتة:", [c for c in probes.columns if probes[c].nunique() == 1])

# 2) الارتباط مع الـ Target مرتبًا
corr = probes[sensors].corrwith(probes[TARGET]).round(3)
display(corr.sort_values(key=abs, ascending=False).to_frame("corr_with_target"))

# 3) الحساسان المكرران
print("corr(temperature_c, temperature_f) =",
      round(probes["temperature_c"].corr(probes["temperature_f"]), 3))

# 4) المتوسط حسب وجود الحياة
means = probes.groupby(TARGET)[sensors].mean().round(2).T
means.columns = ["no_life", "life"]
means["diff"] = (means["life"] - means["no_life"]).round(2)
display(means)

nunique:
 probe_id         300
mission_code       1
methane_ppm      160
temperature_c    246
radiation        212
humidity         192
life_detected      2
temperature_f    246
dtype: int64
معرّفات: ['probe_id']
ثابتة: ['mission_code']


,corr_with_target
methane_ppm,0.882
temperature_c,0.643
temperature_f,0.643
humidity,0.397
radiation,0.110


corr(temperature_c, temperature_f) = 1.0


,no_life,life,diff
methane_ppm,5.38,17.04,11.66
temperature_c,-40.31,-14.78,25.53
temperature_f,-40.55,5.39,45.94
radiation,98.74,100.96,2.22
humidity,19.14,25.63,6.49
